In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Dataset statistics**

In [ ]:
import pandas as pd

In [ ]:


train_df = pd.read_csv("train.csv")

In [ ]:
train_df.head()

,Input_ID,Input_Text,Level1_Class,Level2_Target,Level3_Category,Level4_Subcategory,Level5_Bias
0,2815,சரி நீ தேவிடியா பையன் தான் ஆரியனாள் கற்பழிக்க ...,HATE,No Target- Profane,IMPLICIT,Sarcasm,Nil
1,2174,நடுநிலையாக பேசியதற்கு ஆயிரம் நன்றிகள் ஆனந்த் சார்,NON-HATE,Nil,Nil,Nil,Nil
2,486,எங்க போனாலும் கோத்தா கூதில என் சுண்ணிவாசனைமாரா...,HATE,Hate against Men,EXPLICIT,ExplicitMen-Verbal Abusive,Gender
3,876,ஏன் டீ ஒன்ன பார்த்த பூழூ கூட கிலம்பாது,HATE,Hate against Women,EXPLICIT,ExplicitWomen-Verbal Abusive,Nil
4,2584,நிலா சேலையை கட்டி இருக்க கூடாது உனக்கு அது எது...,HATE,Hate against Women,IMPLICIT,Nil,Nil


In [ ]:
val_df = pd.read_csv("validation.csv")

In [ ]:
train_df.head()

,Input_ID,Input_Text,Level1_Class,Level2_Target,Level3_Category,Level4_Subcategory,Level5_Bias
0,2815,சரி நீ தேவிடியா பையன் தான் ஆரியனாள் கற்பழிக்க ...,HATE,No Target- Profane,IMPLICIT,Sarcasm,Nil
1,2174,நடுநிலையாக பேசியதற்கு ஆயிரம் நன்றிகள் ஆனந்த் சார்,NON-HATE,Nil,Nil,Nil,Nil
2,486,எங்க போனாலும் கோத்தா கூதில என் சுண்ணிவாசனைமாரா...,HATE,Hate against Men,EXPLICIT,ExplicitMen-Verbal Abusive,Gender
3,876,ஏன் டீ ஒன்ன பார்த்த பூழூ கூட கிலம்பாது,HATE,Hate against Women,EXPLICIT,ExplicitWomen-Verbal Abusive,Nil
4,2584,நிலா சேலையை கட்டி இருக்க கூடாது உனக்கு அது எது...,HATE,Hate against Women,IMPLICIT,Nil,Nil


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
from torch.utils.data import RandomSampler
from torch.utils.data import SequentialSampler
from transformers import BertTokenizer
import nltk.data
import re
import string
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwords= stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#load the dataset
data = pd.read_csv("train.csv")
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
data.head()

,Input_ID,Input_Text,Level1_Class,Level2_Target,Level3_Category,Level4_Subcategory,Level5_Bias
0,3042,ஷகிலா மேடம் உங்களுக்கு வர பிட்டு பட வாய்ப்பு இ...,HATE,Hate against Women,IMPLICIT,Sarcasm,Nil
1,3331,25 வயசுல வாழ்க்கையே வாழ்ந்து முடிச்சாச்சி எல்ல...,NON-HATE,Nil,Nil,Nil,Nil
2,470,அத பத்தி நீ ஏன்டா கவலை படுற கமட்டு கூ........,HATE,Hate against Men,EXPLICIT,ExplicitMen-Verbal Abusive,Gender
3,2421,முட்டா புண்டங்களே; அந்த பெண்கள் சரியா தான் சொல...,HATE,Hate against Men,EXPLICIT,Nil,Nil
4,2863,"வாயில நுரை தல்லுது ஆனாலும் பேச்சு குறையலையே,,,...",HATE,Hate against Women,IMPLICIT,Sarcasm,Nil


In [ ]:
data=data.fillna('NULL')

In [ ]:
data['Level1_Class'].value_counts()

Level1_Class
HATE        2231
NON-HATE    2015
Nil            9
Name: count, dtype: int64

In [ ]:
data['Level2_Target'].value_counts()

Level2_Target
Nil                    2052
Hate against Women     1025
Hate against Men        608
No Target- Profane      417
Hate against LGBTQ+     144
Men- no profane           9
Name: count, dtype: int64

In [ ]:
data['Level3_Category'].value_counts()

Level3_Category
Nil         2030
IMPLICIT    1258
EXPLICIT     967
Name: count, dtype: int64

In [ ]:
data['Level4_Subcategory'].value_counts()

Level4_Subcategory
Nil                                              2482
ExplicitMen-Verbal Abusive                        320
Sarcasm                                           295
ExplicitWomen-Verbal Abusive                      243
ImplicitWomen-Character assassination             174
ImplicitWomen-Slut-shaming                         98
ImplicitWomen-Dismissal                            82
ImplicitWomen-Body shaming                         77
ImplicitMen-Character assassination                66
ImplicitWomen-Victim Blaming                       53
ExplicitWomen-Sexual Objectification               43
ExplicitMen-Masculinity Attacks                    43
ImplicitWomen-Moral policing                       32
ImplicitWomen-Steriotype                           27
ImplicitLGBTQ-Moral policing                       24
ExplicitWomen-Sexual Threat                        23
ImplicitMen-Mocking Identity                       22
ImplicitMen-Victim Blaming                         18
ImplicitM

In [ ]:
data['Level5_Bias'].value_counts()

Level5_Bias
Nil           3791
Gender         215
Religional     105
Caste           87
Geography       40
Age             13
Regional         4
Name: count, dtype: int64

In [ ]:
# # Randomly sample exactly 500 rows
# data = data.sample(n=5000, random_state=42)
# #data = data.drop(data.index)


In [ ]:
def preprocessing(text):
    text = re.sub('@[a-zA-Z0-9]*', '', str(text))
    #text = re.sub('?[a-zA-Z0-9]*', '', str(text))
    text = "".join([word.lower() for word in text])
    #text = "".join([word for word in text if not word.isdigit()])
    #tokens = re.split('\W+',text)
    #text = " ".join(word for word in text.split() if word not in stopwords)
    return text

In [ ]:
def data_cleaning(data):
    data['clean_text'] = data['Input_Text'].apply(lambda x:preprocessing(x))
    return data
data=data_cleaning(data)

In [ ]:
data

,Input_ID,Input_Text,Level1_Class,Level2_Target,Level3_Category,Level4_Subcategory,Level5_Bias,clean_text
0,662,priya priya காமிடி கூதியானே நல்லா ஊம்பி கத்து...,HATE,Hate against Women,EXPLICIT,ExplicitWomen-Verbal Abusive,Nil,priya priya காமிடி கூதியானே நல்லா ஊம்பி கத்து...
1,2994,அந்த கோஜா போய் தொலைட்டும்,HATE,Hate against LGBTQ+,IMPLICIT,Sarcasm,Nil,அந்த கோஜா போய் தொலைட்டும்
2,4488,"திருநங்கைகளுக்கு உண்டான மரியாதையே போச்சு ,உங்க...",NON-HATE,Nil,Nil,Nil,Nil,"திருநங்கைகளுக்கு உண்டான மரியாதையே போச்சு ,உங்க..."
3,336,அடேய்........ எச்சை பயலே நீ இந்த மாதிரி பேசுறீ...,HATE,Hate against Men,EXPLICIT,ExplicitMen-Verbal Abusive,Nil,அடேய்........ எச்சை பயலே நீ இந்த மாதிரி பேசுறீ...
4,3169,என்னோட வாழ்க்கை நரகமானாலும் பரவால்ல என்னோட அம்...,NON-HATE,Nil,Nil,Nil,Nil,என்னோட வாழ்க்கை நரகமானாலும் பரவால்ல என்னோட அம்...
...,...,...,...,...,...,...,...,...
4250,4143,இதெல்லாம் show க்கு தான் உண்மையான வாழ்கைக்கு...,NON-HATE,Nil,Nil,Nil,Nil,இதெல்லாம் show க்கு தான் உண்மையான வாழ்கைக்கு...
4251,1378,சின்மயிக்களோட புத்தியே அதுதான். அதுங்களுக்கு அ...,HATE,Hate against Women,IMPLICIT,ImplicitWomen-Character assassination,Nil,சின்மயிக்களோட புத்தியே அதுதான். அதுங்களுக்கு அ...
4252,3228,மனிதனுக்கு மனிதன் ஓத்துவராதாம்.,NON-HATE,Nil,Nil,Nil,Nil,மனிதனுக்கு மனிதன் ஓத்துவராதாம்.
4253,1094,"gay marriage ,...இவனுங்க ரெண்டு பேரும் சேர்ந்த...",HATE,Hate against LGBTQ+,IMPLICIT,ImplicitMen-Mocking Identity,Nil,"gay marriage ,...இவனுங்க ரெண்டு பேரும் சேர்ந்த..."


In [ ]:
X = data['clean_text'].values
y_A = data['Level1_Class'].values
y_B = data['Level2_Target'].values
y_C = data['Level3_Category'].values
y_D = data['Level4_Subcategory'].values
y_E = data['Level5_Bias'].values

KeyError: 'clean_text'

In [ ]:
y_A_clean = [str(label) for label in y_A]
y_B_clean = [str(label) for label in y_B]
y_C_clean = [str(label) for label in y_C]
y_D_clean = [str(label) for label in y_D]
y_E_clean = [str(label) for label in y_E]

# from sklearn.preprocessing import LabelEncoder
# le_B = LabelEncoder()
# y_B_encoded = le_B.fit_transform(y_B_clean)

# #train_labels_B = torch.tensor(y_B_encoded, dtype=torch.long)

In [ ]:
# from sklearn.preprocessing import LabelEncoder ############## not  required
from sklearn.preprocessing import LabelEncoder
label_encoder_A = LabelEncoder()
label_encoder_B = LabelEncoder()
label_encoder_C = LabelEncoder()
label_encoder_D = LabelEncoder()
label_encoder_E = LabelEncoder()

y_A_encoded = label_encoder_A.fit_transform(y_A_clean)
y_B_encoded = label_encoder_B.fit_transform(y_B_clean)
y_C_encoded = label_encoder_C.fit_transform(y_C_clean)
y_D_encoded = label_encoder_D.fit_transform(y_D_clean)
y_E_encoded = label_encoder_E.fit_transform(y_E_clean)

In [ ]:
y_A_encoded

array([1, 0, 0, ..., 1, 0, 1])

In [ ]:
y_B_encoded

array([4, 2, 5, ..., 4, 1, 4])

In [ ]:
y_C_encoded

array([4, 4, 4, ..., 4, 4, 4])

In [ ]:
num_classes_A = data['Level1_Class'].nunique()
num_classes_B = data['Level2_Target'].nunique()
num_classes_C = data['Level3_Category'].nunique()
num_classes_D = data['Level4_Subcategory'].nunique()
num_classes_E = data['Level5_Bias'].nunique()

print(f"Task A: {num_classes_A} classes,Task B: {num_classes_B} classes, Task C: {num_classes_C} classes, Task D: {num_classes_D} classes, Task E: {num_classes_E} classes")

Task A: 3 classes,Task B: 6 classes, Task C: 3 classes, Task D: 32 classes, Task E: 7 classes


In [ ]:
from transformers import  AutoTokenizer, AutoModelForSequenceClassification

X_train, X_val,y_A_train,y_A_val, y_B_train, y_B_val, y_C_train, y_C_val, y_D_train, y_D_val, y_E_train, y_E_val = train_test_split(
    X,y_A_encoded,y_B_encoded, y_C_encoded,y_D_encoded,y_E_encoded, test_size=0.1, random_state=42)

tokenizer = BertTokenizer.from_pretrained('l3cube-pune/tamil-bert')


def tokenize_texts(texts):
    return tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

train_encodings = tokenize_texts(X_train.tolist())
val_encodings = tokenize_texts(X_val.tolist())

tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel

class BertMultitaskModel(nn.Module):
    def __init__(self):
        super(BertMultitaskModel, self).__init__()
        self.bert = BertModel.from_pretrained('l3cube-pune/tamil-bert')


        # Task A
        self.ffn_A = nn.Linear(768, 64)
        self.classifier_A = nn.Linear(64, 3)

        # Task B
        self.ffn_B = nn.Linear(768, 64)
        self.classifier_B = nn.Linear(64, 6)

        # Task C
        self.ffn_C = nn.Linear(768, 64)
        self.classifier_C = nn.Linear(64, 3)

        # Task D
        self.ffn_D = nn.Linear(768, 64)
        self.classifier_D = nn.Linear(64, 32)

        # Task E
        self.ffn_E = nn.Linear(768, 64)
        self.classifier_E = nn.Linear(64, 7)

    def forward(self, input_ids, attention_mask):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Use [CLS] token representation
        cls_output = bert_output.pooler_output  # shape: (batch_size, 768)

        # Task A
        ffn_out_A = torch.relu(self.ffn_A(cls_output))
        logits_A = self.classifier_A(ffn_out_A)

        # Task B
        ffn_out_B = torch.relu(self.ffn_B(cls_output))
        logits_B = self.classifier_B(ffn_out_B)

        # Task C
        ffn_out_C = torch.relu(self.ffn_C(cls_output))
        logits_C = self.classifier_C(ffn_out_C)

        # Task D
        ffn_out_D = torch.relu(self.ffn_D(cls_output))
        logits_D = self.classifier_D(ffn_out_D)

        # Task E
        ffn_out_E = torch.relu(self.ffn_E(cls_output))
        logits_E = self.classifier_E(ffn_out_E)

        return logits_A, logits_B, logits_C, logits_D, logits_E



In [ ]:
y_B_train

array([3, 0, 0, ..., 3, 0, 3])

In [ ]:
y_C_train

array([0, 4, 3, ..., 4, 1, 1])

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW


train_labels_A = torch.tensor(y_A_train)
train_labels_B = torch.tensor(y_B_train)
train_labels_C = torch.tensor(y_C_train)
train_labels_D = torch.tensor(y_D_train)
train_labels_E = torch.tensor(y_E_train)

train_dataset = TensorDataset(
    train_encodings['input_ids'], train_encodings['attention_mask'],
      train_labels_A,train_labels_B, train_labels_C, train_labels_D, train_labels_E)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


model = BertMultitaskModel().to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids, attention_mask,labels_A,labels_B, labels_C, labels_D, labels_E= [t.to(device) for t in batch]
        logits_A,logits_B, logits_C, logits_D, logits_E = model(input_ids, attention_mask)

        loss_A = criterion(logits_A, labels_A)
        loss_B = criterion(logits_B, labels_B)
        loss_C = criterion(logits_C, labels_C)
        loss_D = criterion(logits_D, labels_D)
        loss_E = criterion(logits_E, labels_E)

        loss = loss_A+loss_B + loss_C + loss_D + loss_E
        total_loss += loss.item()

        loss.backward()
        optimizer.step()




    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/951M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: l3cube-pune/tamil-bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1, Loss: 6.971711711088816


In [ ]:
#load the test set
test_df= pd.read_csv('/kaggle/input/crypto-2025/CRYPTO_YOUTUBE_TEST.csv')

In [ ]:
test_df

,MAIN,comment_id
0,This video was a waste of time. The things he ...,video_35661
1,Knowledge is power and a low IQ without knowle...,video_109024
2,This happens already on 2019 to 2021 it was 3...,video_219261
3,Putin body leanguage tell me that he not feeli...,video_101166
4,Anyone who buys ANYTHING from the deepweb it d...,video_75932
...,...,...
495,*I have no problem with my current nvestment c...,video_144917
496,why cant i make a photon account bruh the conn...,video_130270
497,Have you checked out the Adaxum presale? This ...,video_66076
498,Its hilarious watching all these predictions....,video_124400


----------------Test set only --------------------

In [ ]:
test_df=test_df.fillna('NULL')

In [ ]:
def data_cleaning(test_df):
    test_df['clean_text'] = test_df['MAIN'].apply(lambda x:preprocessing(x))
    return test_df
test_df=data_cleaning(test_df)

In [ ]:
test_x= test_df['clean_text'].values

In [ ]:
test_encodings = tokenize_texts(test_x.tolist())

----------------- Test set only -----------------------------------

In [ ]:
test_dataset = TensorDataset(
    val_encodings['input_ids'], val_encodings['attention_mask']) #change it with test_encoding for test set

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


model.eval()

all_preds_A = []
all_preds_B = []
all_preds_C = []
all_preds_D = []
all_preds_E = []

In [ ]:
with torch.no_grad():
    for batch in test_loader:
        # Move input tensors to GPU
        input_ids, attention_mask = [t.to(device) for t in batch]

        # Forward pass through the model
        logits_A,logits_B, logits_C, logits_D, logits_E = model(input_ids, attention_mask)

        # Move the outputs back to the CPU before converting to numpy
        preds_A = torch.argmax(logits_A, dim=1).cpu().numpy()
        preds_B = torch.argmax(logits_B, dim=1).cpu().numpy()
        preds_C = torch.argmax(logits_C, dim=1).cpu().numpy()
        preds_D = torch.argmax(logits_D, dim=1).cpu().numpy()
        preds_E = torch.argmax(logits_E, dim=1).cpu().numpy()

        all_preds_A.extend(preds_A)
        all_preds_B.extend(preds_B)
        all_preds_C.extend(preds_C)
        all_preds_D.extend(preds_D)
        all_preds_E.extend(preds_E)

In [ ]:
print(label_encoder_A.classes_)

[0 1 2]


In [ ]:
predicted_labels_A = label_encoder_A.inverse_transform(all_preds_A)
predicted_labels_B = label_encoder_B.inverse_transform(all_preds_B)
predicted_labels_C = label_encoder_C.inverse_transform(all_preds_C)
predicted_labels_D = label_encoder_D.inverse_transform(all_preds_D)
predicted_labels_E = label_encoder_E.inverse_transform(all_preds_E)

In [ ]:
output_df = pd.DataFrame({
    'Level 1': predicted_labels_A,
    'Level 2': predicted_labels_B,
    'Level 3': predicted_labels_C,
    'Level 4': predicted_labels_D,
    'Level 5': predicted_labels_E
})



NameError: name 'predicted_labels_A' is not defined

In [ ]:
output_df

,Level 1,Level 2,Level 3
0,2,1.0,NULL
1,2,1.0,NULL
2,2,0.0,0.0
3,2,1.0,NULL
4,2,1.0,NULL
...,...,...,...
495,2,1.0,NULL
496,2,0.0,1.0
497,2,0.0,1.0
498,2,1.0,NULL


In [ ]:
y_A_val

array([0, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 2, 1, 0, 2,
       2, 2, 2, 0, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2,
       0, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 1, 1, 2, 2, 2, 2, 1, 2, 0, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2,
       2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 0, 2, 2, 2, 2, 2, 2,
       0, 1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2,
       1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2,
       2, 2, 2, 2, 2, 1, 2, 2, 2, 0, 1, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 1, 0, 2, 2, 2, 0, 0, 2, 2, 1, 0, 2, 2, 0,
       2, 2, 2, 1, 2, 2, 2, 2, 2, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2,
       1, 2, 2, 2, 0, 2, 1, 0, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report as a dictionary
report = classification_report(y_A_val, all_preds_A, output_dict=True)

# Print the full classification report
print("Classification Report for A:")
print(classification_report(y_A_val, all_preds_A))

# Extract and format the macro F1 score
macro_f1 = report['macro avg']['f1-score']
print(f"Macro F1-score: {macro_f1:.4f}")

Classification Report for A:
              precision    recall  f1-score   support

           0       0.76      0.42      0.54       223
           1       0.57      0.85      0.69       203

    accuracy                           0.63       426
   macro avg       0.67      0.64      0.61       426
weighted avg       0.67      0.63      0.61       426

Macro F1-score: 0.6135


In [ ]:
Classification Report for A:
              precision    recall  f1-score   support

           0       0.85      0.71      0.77        75
           1       0.00      0.00      0.00         1
           2       0.95      0.97      0.96       424

    accuracy                           0.93       500
   macro avg       0.60      0.56      0.58       500
weighted avg       0.93      0.93      0.93       500

Macro F1-score: 0.5781

In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report as a dictionary
report = classification_report(y_B_val, all_preds_B, output_dict=True)

# Print the full classification report
print("Classification Report for B:")
print(classification_report(y_B_val, all_preds_B))


# Extract and format the macro F1 score
macro_f1 = report['macro avg']['f1-score']
print(f"Macro F1-score: {macro_f1:.4f}")

Classification Report for B:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        17
           1       0.00      0.00      0.00        70
           2       0.00      0.00      0.00        91
           3       0.00      0.00      0.00         1
           4       0.48      1.00      0.65       203
           5       0.00      0.00      0.00        44

    accuracy                           0.48       426
   macro avg       0.08      0.17      0.11       426
weighted avg       0.23      0.48      0.31       426

Macro F1-score: 0.1076


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
Classification Report for B:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       275
           1       0.69      0.68      0.69       126
           2       0.40      0.17      0.24        23
           3       0.83      0.68      0.75        76

    accuracy                           0.77       500
   macro avg       0.68      0.61      0.63       500
weighted avg       0.76      0.77      0.76       500

Macro F1-score: 0.6304

In [ ]:
from sklearn.metrics import classification_report

# Generate the classification report as a dictionary
report = classification_report(y_C_val, all_preds_C, output_dict=True)

# Print the full classification report
print("Classification Report for C:")
print(classification_report(y_C_val, all_preds_C))

# Extract and format the macro F1 score
macro_f1 = report['macro avg']['f1-score']
print(f"Macro F1-score: {macro_f1:.4f}")

Classification Report for C:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       111
           1       0.00      0.00      0.00       114
           2       0.47      1.00      0.64       201

    accuracy                           0.47       426
   macro avg       0.16      0.33      0.21       426
weighted avg       0.22      0.47      0.30       426

Macro F1-score: 0.2137


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [ ]:
Classification Report for C:
              precision    recall  f1-score   support

           0       0.74      0.77      0.75       176
           1       0.76      0.92      0.83        99
           4       0.84      0.74      0.79       225

    accuracy                           0.78       500
   macro avg       0.78      0.81      0.79       500
weighted avg       0.79      0.78      0.78       500

Macro F1-score: 0.7900

In [ ]:
test_df=test_df.drop('Tweet',axis=1)

In [ ]:
test_df=test_df.drop('clean_text',axis=1)

In [ ]:
final_df=pd.concat([test_df,output_df],axis=1)

In [ ]:
final_df

,MAIN,comment_id,clean_text,Level 1,Level 2,Level 3
0,This video was a waste of time. The things he ...,video_35661,this video was a waste of time. the things he ...,2,1.0,NULL
1,Knowledge is power and a low IQ without knowle...,video_109024,knowledge is power and a low iq without knowle...,2,1.0,NULL
2,This happens already on 2019 to 2021 it was 3...,video_219261,this happens already on 2019 to 2021 it was 3...,2,0.0,0.0
3,Putin body leanguage tell me that he not feeli...,video_101166,putin body leanguage tell me that he not feeli...,2,1.0,NULL
4,Anyone who buys ANYTHING from the deepweb it d...,video_75932,anyone who buys anything from the deepweb it d...,2,1.0,NULL
...,...,...,...,...,...,...
495,*I have no problem with my current nvestment c...,video_144917,*i have no problem with my current nvestment c...,2,1.0,NULL
496,why cant i make a photon account bruh the conn...,video_130270,why cant i make a photon account bruh the conn...,2,0.0,1.0
497,Have you checked out the Adaxum presale? This ...,video_66076,have you checked out the adaxum presale? this ...,2,0.0,1.0
498,Its hilarious watching all these predictions....,video_124400,its hilarious watching all these predictions....,2,1.0,NULL


In [ ]:
final_df=final_df.drop(['clean_text'],axis=1)

In [ ]:
final_df

,MAIN,comment_id,Level 1,Level 2,Level 3
0,This video was a waste of time. The things he ...,video_35661,2,1.0,NULL
1,Knowledge is power and a low IQ without knowle...,video_109024,2,1.0,NULL
2,This happens already on 2019 to 2021 it was 3...,video_219261,2,0.0,0.0
3,Putin body leanguage tell me that he not feeli...,video_101166,2,1.0,NULL
4,Anyone who buys ANYTHING from the deepweb it d...,video_75932,2,1.0,NULL
...,...,...,...,...,...
495,*I have no problem with my current nvestment c...,video_144917,2,1.0,NULL
496,why cant i make a photon account bruh the conn...,video_130270,2,0.0,1.0
497,Have you checked out the Adaxum presale? This ...,video_66076,2,0.0,1.0
498,Its hilarious watching all these predictions....,video_124400,2,1.0,NULL


In [ ]:
final_df.to_csv('/kaggle/working/youtube-59-bert.csv')

print("Predictions saved to test_predictions.csv")

Predictions saved to test_predictions.csv


In [ ]:
print(data.columns)

# or if you're using train_df
print(train_df.columns)

Index(['Input_ID', 'Input_Text', 'Level1_Class', 'Level2_Target',
       'Level3_Category', 'Level4_Subcategory', 'Level5_Bias'],
      dtype='object')
Index(['Input_ID', 'Input_Text', 'Level1_Class', 'Level2_Target',
       'Level3_Category', 'Level4_Subcategory', 'Level5_Bias'],
      dtype='object')
